# Flux post-processing chain: sensible heat flux H, IRGA75 runs (2004-2017)

**notebook version**: `1` (29 Oct 2026). Adapted from the IRGA72 notebook `35_FPPC_IRGA72_H_2016-2026.09`.

## ℹ️ About this notebook

Processes the sensible heat flux `H` with the [diive flux post-processing chain](https://diive.readthedocs.io/en/latest/flux_postprocessing.html): quality
flags (L2), outliers (L3.2) and gap-filling with XGBoost and MDS (L4.1). As in FLUXNET2015 and
ONEFlux (Pastorello et al. 2020), `H` has no storage term (L3.1) and no USTAR filter (L3.3), and
there is no partitioning, uncertainty step or energy balance closure correction.
The site, flux, period and all settings are in *Settings*; *Data sources* links the site
documentation, which holds the decisions and periods specific to the site.

`TEST_RUN = True` runs one year (`TEST_YEAR`) and writes its own output file.

***

## 📇 Data sources

The input `30_FLUX_POST-PROCESSING_CHAIN/IRGA75/01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet` in
the external data folder holds the Level-1 fluxes of the open-path IRGA75 (LI-7500) runs and the
meteo data of the site. The opco correction of that file applies to the CO2 flux only; `H` is
used as EddyPro calculated it.

Site documentation (CH-LAE): the processing levels, settings, periods removed by hand and known
limitations are on the [CH-LAE flux product site](https://holukas.github.io/dataset_ch-lae_flux_product/FPC.html), Levels 3.1 to 4.1.

## ⚙️ Settings

In [ ]:
# --- Source ------------------------------------------------------------------------------------
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
SOURCEFILE = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet"

# --- Testing -----------------------------------------------------------------------------------
# TEST_RUN = True processes only TEST_YEAR, a quick check of the whole notebook. A test run writes
# its own output file, so it never overwrites the result of a full run.
TEST_RUN = False
TEST_YEAR = 2004
# Years to process: the full calendar years with IRGA75 fluxes, as in the IRGA75 CO2 notebook 05.
# The input also holds 2018, without any IRGA75 flux, and 2019 with only the stand-in run of
# 17 January to 20 March; neither is gap-filled here.
YEARS = [TEST_YEAR] if TEST_RUN else list(range(2004, 2018))

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "35_FPPC_IRGA75_H_2004-2017"  # same name as this notebook
if TEST_RUN:
    OUTNAME += f"_TEST_{TEST_YEAR}"
OUTPATH = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75"

# --- Site and flux -----------------------------------------------------------------------------
FLUXCOL = 'H'  # sensible heat flux, from the sonic anemometer alone
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389
UTC_OFFSET = 1  # timestamps are UTC+01:00 all year
NIGHTTIME_THRESHOLD = 20  # W m-2 potential radiation below this = nighttime
DAYTIME_ACCEPT_QCF_BELOW = 2  # keep QCF 0 and 1
NIGHTTIME_ACCEPT_QCF_BELOW = 2
LONG_GAP_DAYS = 2  # gaps in the measured H at least this long are listed after loading

# --- Level 2 -----------------------------------------------------------------------------------
# The tests of the gas analyser (signal strength, gas completeness) do not apply to H. The raw-data
# tests read T_SONIC_VM97_TEST, the column of the base variable of H. Its absolute-limits test is
# added: a sonic temperature out of range points to water or ice on the transducers.
LEVEL2_SETTINGS = {
    'signal_strength': {'apply': False},
    'raw_data_screening_vm97': {'apply': True,
                                'spikes': True, 'amplitude': False, 'dropout': True,
                                'abslim': True, 'skewkurt_hf': False, 'skewkurt_sf': False,
                                'discont_hf': False, 'discont_sf': False},
    'ssitc': {'apply': True, 'setflag_timeperiod': None},
    'gas_completeness': {'apply': False},
    'spectral_correction_factor': {'apply': True},
    'angle_of_attack': {'apply': False, 'application_dates': []},
    'steadiness_of_horizontal_wind': {'apply': False},
}

# --- Level 3.2 ---------------------------------------------------------------------------------
# Periods removed by hand (date ranges, whole days, both ends included), with the reason for each.
REMOVE_DATES = []  # none for the IRGA75 runs so far
# Absolute limits (W m-2), chosen from the data: the upper limit keeps the largest daytime values
# that agree with SW_IN, the lower limit lies below the nighttime values of summer and removes the
# large negative values of wet and freezing episodes in winter.
ABSLIM = (-200, 800)

# --- Level 3.3 ---------------------------------------------------------------------------------
# No USTAR filter for H (Pastorello et al. 2020). Level 4.1 needs a Level-3.3 series, so the chain
# runs one scenario with threshold 0, which removes no record.
USTAR_THRESHOLDS = [0]
USTAR_SCENARIOS = ['CUT_NONE']
SCENARIO = USTAR_SCENARIOS[0]

# --- Level 4.1: meteo drivers ------------------------------------------------------------------
TA_F = 'TA_T1_47_1_HOMOGENIZED_gfXG'    # degC, complete
SW_IN_F = 'SW_IN_T1_47_1_gfXG'          # W m-2, complete
VPD_F = 'VPD_T1_47_1_HOMOGENIZED'       # kPa, complete
TA_FLAG = 'FLAG_TA_T1_47_1_ISFILLED'    # 0 = measured at 47 m
SW_IN_FLAG = 'FLAG_SW_IN_T1_47_1_ISFILLED'
VPD_FLAG = 'FLAG_VPD_T1_47_1_ISFILLED'
FEATURES = [TA_F, SW_IN_F, VPD_F]

FLUX_UNITS = 'W m-2'

# 30-min energy flux in W m-2 -> MJ m-2 per half-hour, for sums and cumulatives.
WM2_TO_MJ = 1800 * 1e-6

# MDS gap-filling, the method of FLUXNET2015 for H, next to XGBoost. mode='diive' fills every gap,
# also long ones; the QC flags keep model-filled driver values out of the similar samples.
RUN_MDS = True
MDS_SETTINGS = dict(mode='diive', fill_long_gaps=True)

# XGBoost hyperparameters, at the production values suggested by the diive notebook.
XGB_SETTINGS = dict(n_estimators=1000, max_depth=6, learning_rate=0.05,
                    random_state=42, n_jobs=-1, shap_max_rows=5000)

In [ ]:
from datetime import datetime
import importlib.metadata
import warnings

import diive as dv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from diive.core.ml.feature_engineer import FeatureEngineer
from diive.core.plotting.styles import chainplots as cp
from diive.core.times.times import doy_cumulatives_per_year, doy_mean_cumulative
from diive.flux.postprocessingchain import (
    init_flux_data,
    make_level32_detector,
    run_level2,
    run_level31,
    run_level32,
    run_level33_constant_ustar,
    run_level41_mds,
    run_level41_xgb,
    # Reporting over the finished chain
    plot_feature_ranks_per_year,
    plot_tree_selection,
    plot_gapfilled_cumulative,
    plot_gapfilling_comparison,
    plot_gapfilling_input,
    plot_quality_heatmaps,
    plot_quality_timeseries,
    report_annual_sums,
    report_gapfilling_comparison,
    report_gapfilling_feature_importances,
    report_gapfilling_input,
    report_gapfilling_model_scores,
    report_gapfilling_poolyears,
    report_gapfilling_variables,
    report_gaps,
    report_model_overview,
    report_traintest_model_scores,
    report_tree_selection,
)

# The notebook's inline backend renders each figure itself; diive's own fig.show() call can only
# warn that it cannot open a window.
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')

# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

## 📂 Load data

`init_flux_data` adds `SW_IN_POT`, `DAYTIME` and `NIGHTTIME` itself and stops if they already exist,
so they are removed from the input first.

In [ ]:
df = dv.load_parquet(filepath=SOURCEFILE)
df = df.drop(columns=[c for c in ('SW_IN_POT', 'DAYTIME', 'NIGHTTIME') if c in df.columns])
df = df.loc[df.index.year.isin(YEARS)].copy()
assert sorted(set(df.index.year)) == sorted(YEARS), f"the input lacks some of the years {YEARS}"
assert (df.index[0], df.index[-1]) == (pd.Timestamp(f'{min(YEARS)}-01-01 00:15'),
                                       pd.Timestamp(f'{max(YEARS)}-12-31 23:45')), \
    f"the selected years run from {df.index[0]} to {df.index[-1]}, not over whole calendar years"

for _c in [FLUXCOL, 'FC', TA_F, SW_IN_F, VPD_F, TA_FLAG, SW_IN_FLAG, VPD_FLAG,
           f'{FLUXCOL}_SSITC_TEST', f'{FLUXCOL}_SCF', 'T_SONIC_VM97_TEST']:
    assert _c in df.columns, f"{_c} is not in {SOURCEFILE}"
for _c in FEATURES:
    assert df[_c].notna().all(), f"driver {_c} has gaps"
print(f"{len(df):,} records x {df.shape[1]} columns, {df.index[0]} -> {df.index[-1]}")
print(f"{FLUXCOL}: {df[FLUXCOL].notna().sum():,} records")

### 📋 Table: records of H per year

The measured `H` of each year before any test, the days without a single `H` record, and the
records of `H` that have no `FC`, mostly from periods in which only the gas analyser failed.

In [ ]:
_h = df[FLUXCOL].notna()
_year = df.index.year
_days = _h.groupby(df.index.normalize()).sum()
COVERAGE = pd.DataFrame({
    'records': _h.groupby(_year).size(),
    'measured': _h.groupby(_year).sum(),
    'measured %': (100 * _h.groupby(_year).mean()).round(1),
    'days without H': (_days == 0).groupby(_days.index.year).sum(),
    'without FC': (_h & df['FC'].isna()).groupby(_year).sum(),
})
COVERAGE.index.name = 'year'
display(COVERAGE)

### 📋 Table: long gaps in H

Every period of at least `LONG_GAP_DAYS` days without a measured `H` in the input, including one at
the start or end of the processed period.

In [ ]:
# Runs of records without H: a new run starts wherever H changes between measured and missing.
_miss = ~_h
_run = (_miss != _miss.shift()).cumsum()[_miss]
_g = df.index.to_series()[_miss].groupby(_run)
LONG_GAPS = pd.DataFrame({'first missing': _g.min(), 'last missing': _g.max(), 'records': _g.size()})
LONG_GAPS['days'] = (LONG_GAPS['records'] / 48).round(1)
LONG_GAPS = LONG_GAPS[LONG_GAPS['days'] >= LONG_GAP_DAYS].reset_index(drop=True)
display(LONG_GAPS)

## Step 1: set up the data

Calculates the potential radiation, marks day and night, and stores the site coordinates that later
steps use. `*_ACCEPT_QCF_BELOW = 2` keeps records with quality flag 0 (all tests passed) and 1
(minor warnings).

In [ ]:
data = init_flux_data(
    df=df,
    fluxcol=FLUXCOL,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=UTC_OFFSET,
    nighttime_threshold=NIGHTTIME_THRESHOLD,
    daytime_accept_qcf_below=DAYTIME_ACCEPT_QCF_BELOW,
    nighttime_accept_qcf_below=NIGHTTIME_ACCEPT_QCF_BELOW,
)
print(data)

## Step 2: quality flags (Level 2)

The tests in `LEVEL2_SETTINGS`: steady state and developed turbulence (SSITC), the spectral
correction factor, and spikes, dropouts and absolute limits in the raw sonic temperature (Vickers
and Mahrt 1997). The tests of the gas analyser do not apply to `H`. The table shows how many records
each test removes.

In [ ]:
data = run_level2(data, **LEVEL2_SETTINGS)
print(f'After L2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: Level-2 quality flag over time

In [ ]:
data.levels.level2_qcf.showplot_qcf_heatmaps()

### 📋 Table: result of each Level-2 test

In [ ]:
data.levels.level2_qcf.report_qcf_flags()

## Step 3: no storage term (Level 3.1)

`H` has no storage term, as in FLUXNET2015 and ONEFlux (Pastorello et al. 2020). Levels 3.2 and
3.3 need Level 3.1, so `run_level31` runs with `set_storage_to_zero=True`: the column `H_L3.1`
equals `H` ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#storage-term)).

In [ ]:
data = run_level31(data, set_storage_to_zero=True, gapfill_storage_term=False)
print(f'After L3.1: {data.filteredseries.dropna().count():,} valid records '
      f'(col: {data.filteredseries.name})')

## Step 4: outlier removal (Level 3.2)

Four tests. Check each plot, then accept the test with `sod.addflag()` in the cell below it.

1. Periods in `REMOVE_DATES`, removed by hand; the reason for each is given in *Settings*.
2. Absolute limits `ABSLIM`.
3. Hampel filter on double differences: single spikes.
4. Hampel filter per time of day: values wrong for their time of day.

Negative nighttime values are not tested: a negative `H` at night is physical. How the tests work:
[diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#outlier-tests). The table at the end of this step lists further tests.

In [ ]:
data, sod = make_level32_detector(data)

### 📊 Plot: periods removed by hand

In [ ]:
# Only the date ranges that overlap the processed records; none means no test and no flag.
_first, _last = data.full_df.index[0], data.full_df.index[-1]
_remove = [_d for _d in REMOVE_DATES
           if pd.Timestamp(_d[0]) <= _last and pd.Timestamp(_d[1]) + pd.Timedelta('1D') > _first]
if _remove:
    sod.flag_manualremoval_test(remove_dates=_remove, showplot=True, verbose=True)
else:
    print(f"No manual removal dates in {_first:%Y-%m-%d} - {_last:%Y-%m-%d}; test skipped.")

In [ ]:
if _remove:
    sod.addflag()  # accept the manual removal

### 📊 Plot: absolute limits

In [ ]:
sod.flag_outliers_abslim_test(minval=ABSLIM[0], maxval=ABSLIM[1], showplot=True, verbose=True)

In [ ]:
sod.addflag()  # accept this test

### 📊 Plot: Hampel filter on double differences

In [ ]:
sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=True,
                              separate_day_night=True, showplot=True, verbose=True, repeat=True)

In [ ]:
sod.addflag()  # accept this test

### 📊 Plot: Hampel filter on values per time of day

In [ ]:
sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=False,
                              by_time_of_day=True, time_of_day_slots=1, min_records=10,
                              separate_day_night=True, showplot=True, verbose=True, repeat=True)

In [ ]:
sod.addflag()  # accept this test

### 📋 Table: outlier tests available in diive

Every test of the outlier detector `sod`, with the ones used above marked. `help(sod.<method>)`
shows the settings of a test.

In [ ]:
_used = {'flag_manualremoval_test', 'flag_outliers_abslim_test', 'flag_outliers_hampel_test'}
_rows = []
for _name in sorted(dir(type(sod))):
    if not (_name.startswith('flag_') and _name.endswith('_test')):
        continue
    # The first sentence of the docstring, which may run over two lines.
    _doc = (getattr(type(sod), _name).__doc__ or '').strip().split('\n\n')[0]
    _first = ' '.join(_doc.split()).split('. ')[0].rstrip('.')
    _rows.append({'method': f'sod.{_name}()', 'what it does': _first,
                  'used here': 'yes' if _name in _used else ''})
display(pd.DataFrame(_rows).set_index('method'))

### Apply the Level-3.2 flags

In [ ]:
data = run_level32(data, outlier_detector=sod)
print(f'After L3.2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: all removed outliers

Every record Level 3.2 removed, coloured by the test that removed it, with the share removed per
month and per hour of the day.

In [ ]:
data.levels.level32.showplot_overview();

### 📊 Plot: Level-3.2 quality flag over time

In [ ]:
data.levels.level32_qcf.showplot_qcf_heatmaps()

### 📋 Table: result of each Level-3.2 test

In [ ]:
data.levels.level32_qcf.report_qcf_flags()

## Step 5: no USTAR filter (Level 3.3)

No USTAR filter for `H`, as in FLUXNET2015 and ONEFlux (Pastorello et al. 2020). Level 4.1 reads
Level 3.3, so the chain runs one scenario, `CUT_NONE`, with threshold 0, which removes no record.
Column names carry `_L3.3_CUT_NONE_` ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#ustar-filter)).

In [ ]:
data = run_level33_constant_ustar(
    data,
    thresholds=USTAR_THRESHOLDS,
    threshold_labels=USTAR_SCENARIOS,
    showplot=False,
    verbose=True,
)

### Records for gap-filling and highest-quality records

The records the gap-filling starts from (QCF 0 and 1, columns `_QCF`) and the highest-quality
records (QCF 0, columns `_QCF0`) for analyses without gap-filled values
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#gapfilling-input)).

### 📋 Table: records for gap-filling

In [ ]:
report_gapfilling_input(data);

### 📊 Plot: records for gap-filling and highest-quality records

In [ ]:
plot_gapfilling_input(data, units=FLUX_UNITS)

### 📋 Table: gaps to fill

The gaps left after Level 3.3, which the gap-filling has to fill. A long gap lasts at least one day.

In [ ]:
report_gaps(data, long_gap_days=1);

## Step 6: model features

The feature engineer adds lagged values, rolling statistics and time terms to the drivers. It is built once and passed to XGBoost. `target_col` is only a placeholder.

In [ ]:
engineer = FeatureEngineer(
    target_col='_target_',
    features_lag=[-2, 2],
    features_lag_stepsize=1,
    features_rolling=[2, 4, 12, 24, 48],
    features_rolling_stats=['median', 'min', 'max', 'std'],
    features_ema=[6, 12, 24, 48],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    verbose=0,
)

## Step 7: gap-filling (Level 4.1)

XGBoost and MDS fill the gaps; MDS is the method of FLUXNET2015 and ONEFlux for the energy fluxes
(Pastorello et al. 2020). Step 10 compares the two.

### XGBoost

One model per year, trained on three years, with features selected by SHAP
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#xgboost-gapfilling)).

The number of trees is chosen once for all models by blocked cross-validation
([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#tree-selection)).

In [ ]:
data = run_level41_xgb(
    data,
    features=FEATURES,
    engineer=engineer,
    reduce_features=True,
    n_estimators_cv=True,
    verbose=1,
    **XGB_SETTINGS,
)

### MDS

Marginal distribution sampling (Reichstein et al. 2005) in `mode='diive'`, filling every gap from
measured driver values only ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#mds-gapfilling)).

**Skipped** when `RUN_MDS = False`.

In [ ]:
if RUN_MDS:
    data = run_level41_mds(
        data,
        swin=SW_IN_F, ta=TA_F, vpd=VPD_F,  # W m-2, degC, kPa
        swin_qc=SW_IN_FLAG, ta_qc=TA_FLAG, vpd_qc=VPD_FLAG,
        **MDS_SETTINGS,
    )

## Step 8: results

How many records were measured and how many were filled, then the annual sum of the flux. The
share measured is given in brackets: a year with long gaps rests largely on the model. An incomplete
year is marked.

### 📋 Table: measured and filled records

In [ ]:
report_gapfilling_variables(data)

### 📋 Table: annual sums of the flux

In [ ]:
report_annual_sums(data, gain=WM2_TO_MJ, units='MJ m-2');

## Step 9: model quality

First the scores on test data: a random 25 % of the complete records, which the model did not see in
training. The number of trees is chosen without the test data. Because the test
records lie between training records, the scores describe short gaps better than gaps of several
days.

### 📋 Table: scores on test data

In [ ]:
report_traintest_model_scores(data)

### 📋 Table: gap-filling models per year

How each model was built and how well it did: the years it was trained on, its features, the
training and test records, the number of trees, the records of the final fit,
the test scores, and how the gaps of its year were filled.

In [ ]:
report_model_overview(data);

### 📋 Table: number of trees from blocked cross-validation

The scores on held-out blocks of days at the chosen number of trees, next to the scores on the
random test records. The blocked scores describe gaps of days, the random ones single missing
records ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#tree-selection)).

In [ ]:
report_tree_selection(data);

### 📊 Plot: number of trees from blocked cross-validation

The error on held-out blocks against the number of trees, one line per pool, and the chosen
number. The curves should flatten well before the largest number tested.

In [ ]:
plot_tree_selection(data);

Next the scores on all measured records. The gaps are filled by a model refitted on all of them, so
these scores are better than the test scores and only show how closely the model fits.

### 📋 Table: scores on all measured records

In [ ]:
report_gapfilling_model_scores(data)

### 📋 Table: years used to train each year's model

In [ ]:
report_gapfilling_poolyears(data)

### 📋 Table: most important features (SHAP)

In [ ]:
report_gapfilling_feature_importances(data, ustar_scenario=SCENARIO)

### 📊 Plot: feature ranks per year

In [ ]:
plot_feature_ranks_per_year(data)

## Step 10: result plots

### 📊 Plot: flux by quality class, heatmaps, XGBoost

The highest-quality records (QCF 0), the records the gap-filling used (QCF 0 and 1), the rejected
records, and the gap-filled result, on one colour scale.

In [ ]:
plot_quality_heatmaps(data, ustar_scenario=SCENARIO, units=FLUX_UNITS, method='xgb')

### 📊 Plot: flux by quality class, day and night, XGBoost

Daytime and nighttime records by quality class, then the gap-filled result, with measured and
filled records shown apart.

In [ ]:
plot_quality_timeseries(data, ustar_scenario=SCENARIO, units=FLUX_UNITS, method='xgb')

### 📊 Plot: measured and gap-filled flux

In [ ]:
data.plot_gapfilled_heatmaps(ustar_scenario=SCENARIO)

### 📊 Plot: cumulative flux

In [ ]:
data.plot_cumulative_comparison(
    ustar_scenario=SCENARIO,
    conv_factor=WM2_TO_MJ,
    units='MJ m-2',
)

### 📊 Plot: cumulative flux of both gap-filling methods

One line per gap-filling method, each in its own colour, over the whole processed period.

In [ ]:
plot_gapfilled_cumulative(
    data,
    gain=WM2_TO_MJ,
    units='MJ m-2',
    per_year=False,
)

### 📊 Plot: cumulative flux per year

One panel per gap-filling method, on the same y-axis. One line per year; the band is the mean ± 1 SD
of the full years. An incomplete last year is left out of the band.

In [ ]:
# One panel per gap-filling method, on a shared y-axis. Years run from light to dark, so the most
# recent year is the darkest line.
_gf = data.gapfilled_cols()
_names = {'mds': 'MDS', 'xgb': 'XGBoost', 'rf': 'random forest'}
_months = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
# A last year that ends before 31 December stays out of the mean and the band.
_end = data.fpc_df.index[-1]
_partial = [_end.year] if (_end.month, _end.day) != (12, 31) else None
fig, axes = plt.subplots(1, len(_gf), figsize=(8 * len(_gf), 6.6), sharex=True, sharey=True,
                         layout='constrained', squeeze=False)
for _ax, (_method, _cols) in zip(axes[0], _gf.items()):
    _cum = doy_cumulatives_per_year(data.fpc_df[_cols[SCENARIO]] * WM2_TO_MJ)
    _cum = _cum.loc[:, _cum.notna().any()]
    _years = list(_cum.columns)
    _shades = plt.get_cmap(cp.METHOD_CMAPS.get(_method, 'Greys'))(
        np.linspace(0.35, 1.0, len(_years)) if len(_years) > 1 else [0.85])
    if len(_years) > 1:
        _ref = doy_mean_cumulative(cumulatives_per_year_df=_cum,
                                   excl_years_from_reference=_partial)
        # Day 366 exists only in leap years, so the band stops before it.
        _ref = _ref.loc[_ref.index < 366]
        # The legend gives the mean and SD of the annual totals of the full years.
        _totals = [_cum[_y].dropna().iloc[-1] for _y in _years if _y not in (_partial or [])]
        _end_mean, _end_sd = np.mean(_totals), np.std(_totals, ddof=1)
        _ax.fill_between(_ref.index, _ref['MEAN-SD'], _ref['MEAN+SD'], color=cp.KEPT,
                         alpha=0.3, linewidth=0, label=f'mean ± 1 SD: ±{_end_sd:,.0f}')
        _ax.plot(_ref.index, _ref['MEAN_DOY_TIME'], color=cp.INK, linewidth=1.2,
                 linestyle=(0, (4, 3)), label=f'mean: {_end_mean:,.0f}')
    for _shade, _year in zip(_shades, _years):
        _valid = _cum[_year].dropna()
        _ax.plot(_cum.index, _cum[_year], color=_shade, linewidth=1.5,
                 label=f"{_year}: {_valid.iloc[-1]:,.0f}")
    _ax.axhline(0, color=cp.INK, linewidth=0.8, alpha=0.5)
    cp.style_axes(_ax)
    cp.panel_label(_ax, _names.get(_method, _method))
    _ax.set_xticks(_months, ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    _ax.set_xlim(1, 367)
    _ax.legend(frameon=False, fontsize=8, ncol=1 if len(_years) < 8 else 2)
axes[0][0].set_ylabel('cumulative flux (MJ m-2)')
_subtitle = "One panel per gap-filling method; the total of each year in the legend."
if _partial:
    _subtitle += f" {_end.year} ends on {_end:%d %B} and is left out of the mean and the band."
cp.figure_title(fig, 'Cumulative flux per year', _subtitle)
plt.show()

### Gap-filling methods compared

The difference between the sums of MDS and XGBoost comes from the filled gaps alone; the table
splits it by day and night and by gap length ([diive](https://diive.readthedocs.io/en/latest/flux_postprocessing.html#method-comparison)).

**Skipped** when `RUN_MDS = False`.

#### 📋 Table: MDS and XGBoost, sums and their difference

In [ ]:
if RUN_MDS:
    report_gapfilling_comparison(data, methods=('mds', 'xgb'), gain=WM2_TO_MJ, units='MJ m-2');

#### 📊 Plot: where MDS and XGBoost differ

The filled values of both methods at the same gaps, their mean difference by month and hour, the
contribution of each month to the difference of the sums, and the cumulative difference.

In [ ]:
if RUN_MDS:
    plot_gapfilling_comparison(data, ustar_scenario=SCENARIO, methods=('mds', 'xgb'),
                               gain=WM2_TO_MJ, units='MJ m-2', flux_units=FLUX_UNITS);

## 💾 Step 11: export

The output file holds only the columns this notebook adds: the flags and series of every level and
the gap-filled flux. The input columns stay in the input file; the merge into the fullset joins the
flux outputs with it on the timestamp.

In [ ]:
_all = data.merged_df()
out_df = _all[[c for c in _all.columns if c not in df.columns]]
assert out_df.index.equals(df.index), "the output timestamps differ from the input"
print(f"{out_df.shape[0]:,} records x {out_df.shape[1]} columns "
      f"(input columns left out: {_all.shape[1] - out_df.shape[1]})")
filepath = dv.save_parquet(data=out_df, filename=OUTNAME, outpath=OUTPATH)

***

# End of notebook.

In [ ]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")